# lean v4 — self-distillation, not gold-label compression

v1/v2/v3 trained on GSM8K's own gold-label style (uniformly ~52 words
regardless of difficulty) and landed 45-51% accuracy vs base's 69%.

v4 trains on `pairs_v4.jsonl` instead: the base model's own reasoning,
kept only where it was already verified correct (5,780 of 7,473, 77.3%),
with boilerplate intro/closing trimmed but the actual derivation untouched
(26.5% shorter than the raw generations, 0% of that trim broke a correct
answer). Length still varies by difficulty (unlike v1-v3's fixed template).

1. New Notebook → Settings → Accelerator: **GPU T4 x2** (or P100).
2. Add Data → upload `data/pairs_v4.jsonl` as a new Kaggle dataset (e.g. `lean-pairs-v4`).
3. Run all cells.

In [ ]:
!pip install -q unsloth trl peft datasets

In [ ]:
CFG = {
    "base_model": "unsloth/Qwen2.5-1.5B-Instruct-bnb-4bit",
    "seed": 0,
    "data_path": "/kaggle/input/datasets/johnandreimartinez/lean-pairs-v4/pairs_v4.jsonl",  # adjust to your uploaded dataset
    "output_dir": "/kaggle/working/lean-v4",
    "epochs": 2,
    "batch_size": 2,
    "grad_accum": 8,
    "lr": 2.0e-4,
    "max_seq_len": 1024,
    "lora": {
        "r": 16, "alpha": 16, "dropout": 0,
        "target_modules": ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    },
}

In [ ]:
from datasets import load_dataset
from trl import SFTConfig, SFTTrainer
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    CFG["base_model"], max_seq_length=CFG["max_seq_len"], load_in_4bit=True,
)
model = FastLanguageModel.get_peft_model(
    model,
    r=CFG["lora"]["r"],
    lora_alpha=CFG["lora"]["alpha"],
    lora_dropout=CFG["lora"]["dropout"],
    target_modules=CFG["lora"]["target_modules"],
    random_state=CFG["seed"],
)

In [ ]:
ds = load_dataset("json", data_files=CFG["data_path"], split="train")
ds = ds.map(lambda r: {"text": f"{r['prompt']}\n\n{r['completion']}{tokenizer.eos_token}"})

In [ ]:
trainer = SFTTrainer(
    model=model,
    processing_class=tokenizer,
    train_dataset=ds,
    args=SFTConfig(
        output_dir=CFG["output_dir"],
        num_train_epochs=CFG["epochs"],
        per_device_train_batch_size=CFG["batch_size"],
        gradient_accumulation_steps=CFG["grad_accum"],
        learning_rate=CFG["lr"],
        max_length=CFG["max_seq_len"],
        seed=CFG["seed"],
        logging_steps=10,
        save_strategy="epoch",
        fp16=True,
        bf16=False,
        dataset_text_field="text",
    ),
)
trainer.train()

In [ ]:
model.save_pretrained(CFG["output_dir"] + "/adapter")
tokenizer.save_pretrained(CFG["output_dir"] + "/adapter")
# Download /kaggle/working/lean-v4/adapter as Kaggle output, then run eval.py locally.